In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import argparse

class VGG7(nn.Module):
    def __init__(self):
        super(VGG7, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256*4*4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

class CIFAR10Classifier:
    def __init__(self, args):
        self.args = args
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.classes = ('plane', 'car', 'bird', 'cat', 'deer', 
                       'dog', 'frog', 'horse', 'ship', 'truck')
        
        self.model = VGG7().to(self.device)
        self.criterion = nn.CrossEntropyLoss()
        
        if args.optimizer.lower() == 'adam':
            self.optimizer = optim.Adam(self.model.parameters(), 
                                      lr=args.lr, 
                                      weight_decay=args.weight_decay)
        elif args.optimizer.lower() == 'sgd':
            self.optimizer = optim.SGD(self.model.parameters(), 
                                      lr=args.lr, 
                                      momentum=0.9,
                                      weight_decay=args.weight_decay)
        else:
            raise ValueError(f"Unsupported optimizer: {args.optimizer}")
        
        self._load_data()

    def _load_data(self):
        transform_train = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ])

        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ])

        trainset = torchvision.datasets.CIFAR10(
            root='./data', train=True, download=True, transform=transform_train)
        self.trainloader = DataLoader(trainset, batch_size=self.args.batch_size,
                                     shuffle=True, num_workers=2)

        testset = torchvision.datasets.CIFAR10(
            root='./data', train=False, download=True, transform=transform_test)
        self.testloader = DataLoader(testset, batch_size=self.args.batch_size,
                                    shuffle=False, num_workers=2)

    def train(self):
        print(f"\nStarting training with {self.args.epochs} epochs...")
        filename = f"opt_{self.args.optimizer}_lr{self.args.lr}_wd{self.args.weight_decay}_bs{self.args.batch_size}_ep{self.args.epochs}-acc.txt"
        
        with open(filename, 'w') as f:  
            f.write("Epoch, Train_acc, Test_acc\n")     
            
            for epoch in range(self.args.epochs):
                self.model.train()      
                total = 0
                correct = 0
                
                for inputs, targets in self.trainloader:
                    inputs, targets = inputs.to(self.device), targets.to(self.device)
                    
                    self.optimizer.zero_grad()
                    outputs = self.model(inputs)
                    loss = self.criterion(outputs, targets)
                    loss.backward()
                    self.optimizer.step()
                    
                    _, predicted = outputs.max(1)
                    total += targets.size(0)
                    correct += predicted.eq(targets).sum().item()
                
                train_acc = 100 * correct / total
                
                test_acc = self.test()
                
                f.write(f"{epoch}, {train_acc:.2f}, {test_acc:.2f}\n") 
                
                if (epoch+1) % 10 == 0:
                    print(f"Epoch [{epoch+1}/{self.args.epochs}] | "
                          f"Train Acc: {train_acc:.2f}% | "
                          f"Test Acc: {test_acc:.2f}%")
        
        model_save_name = f"opt_{self.args.optimizer}_lr{self.args.lr}_wd{self.args.weight_decay}_bs{self.args.batch_size}_ep{self.args.epochs}.pth"
        torch.save(self.model.state_dict(), model_save_name)
        print(f"Model saved to {model_save_name}")

    def test(self):
        self.model.eval()
        all_targets = []
        all_preds = []
        with torch.no_grad():
            for inputs, targets in self.testloader:
                inputs, targets = inputs.to(self.device), targets.to(self.device)
                outputs = self.model(inputs)
                _, predicted = outputs.max(1)
                
                all_targets.extend(targets.cpu().numpy())
                all_preds.extend(predicted.cpu().numpy())
        
        accuracy = 100 * np.sum(np.array(all_preds) == np.array(all_targets)) / len(all_targets)
        return accuracy

    def plot_confusion_matrix(self):
        self.model.eval()
        all_targets = []
        all_preds = []
        
        with torch.no_grad():
            for inputs, targets in self.testloader:
                inputs, targets = inputs.to(self.device), targets.to(self.device)
                outputs = self.model(inputs)
                _, predicted = outputs.max(1)
                
                all_targets.extend(targets.cpu().numpy())
                all_preds.extend(predicted.cpu().numpy())
        
        cm = confusion_matrix(all_targets, all_preds)
        plt.figure(figsize=(12, 10))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=self.classes, 
                   yticklabels=self.classes)
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        
        filename = f"confusion_matrix_opt_{self.args.optimizer}_lr{self.args.lr}_wd{self.args.weight_decay}_bs{self.args.batch_size}_ep{self.args.epochs}.png"
        plt.savefig(filename)
        plt.close() 
        print(f"Confusion matrix saved to {filename}")

In [2]:
from types import SimpleNamespace

param_grid = {
    'batch_size': [64, 128],
    'lr': [ 0.0001, 0.00001],
    'weight_decay': [5e-4, 5e-5],
    'optimizer': ['sgd', 'adam']
}

base_args = {
    'train': True,
    'test': True,
    'plot': True,
    'epochs': 50,
    'save_path': None,
    'load_path': None
}

results = []

for batch_size in param_grid['batch_size']:
    for lr in param_grid['lr']:
        for weight_decay in param_grid['weight_decay']:
            for optimizer in param_grid['optimizer']:
                print(f"\n=== Testing config: batch_size={batch_size}, lr={lr}, "
                      f"weight_decay={weight_decay}, optimizer={optimizer} ===")
                
                current_args = base_args.copy()
                current_args.update({
                    'batch_size': batch_size,
                    'lr': lr,
                    'weight_decay': weight_decay,
                    'optimizer': optimizer,
                    'save_path': f"model_bs{batch_size}_lr{lr}_wd{weight_decay}_{optimizer}.pth"
                })
                
                args = SimpleNamespace(**current_args)
                
                classifier = CIFAR10Classifier(args)
                
                if args.train:
                    classifier.train()
                
                if args.test:
                    accuracy = classifier.test()
                    results.append({
                        'batch_size': batch_size,
                        'lr': lr,
                        'weight_decay': weight_decay,
                        'optimizer': optimizer,
                        'accuracy': accuracy
                    })
                    print(f"Test Accuracy: {accuracy:.2f}%")

if results:
    best_result = max(results, key=lambda x: x['accuracy'])
    print("\n=== Best Configuration ===")
    print(f"Batch Size: {best_result['batch_size']}")
    print(f"Learning Rate: {best_result['lr']}")
    print(f"Weight Decay: {best_result['weight_decay']}")
    print(f"Optimizer: {best_result['optimizer']}")
    print(f"Accuracy: {best_result['accuracy']:.2f}%")

    with open('best_config.txt', 'w') as f:
        f.write(str(best_result))


=== Testing config: batch_size=64, lr=0.0001, weight_decay=0.0005, optimizer=sgd ===


100%|██████████| 170M/170M [00:10<00:00, 15.8MB/s]


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified

Starting training with 50 epochs...
Epoch [10/50] | Train Acc: 35.77% | Test Acc: 41.36%
Epoch [20/50] | Train Acc: 49.61% | Test Acc: 55.49%
Epoch [30/50] | Train Acc: 58.04% | Test Acc: 62.12%
Epoch [40/50] | Train Acc: 63.47% | Test Acc: 64.99%
Epoch [50/50] | Train Acc: 67.54% | Test Acc: 70.20%
Model saved to opt_sgd_lr0.0001_wd0.0005_bs64_ep50.pth
Test Accuracy: 70.20%

=== Testing config: batch_size=64, lr=0.0001, weight_decay=0.0005, optimizer=adam ===
Files already downloaded and verified
Files already downloaded and verified

Starting training with 50 epochs...
Epoch [10/50] | Train Acc: 79.29% | Test Acc: 80.30%
Epoch [20/50] | Train Acc: 86.10% | Test Acc: 84.03%
Epoch [30/50] | Train Acc: 89.75% | Test Acc: 85.71%
Epoch [40/50] | Train Acc: 91.86% | Test Acc: 86.52%
Epoch [50/50] | Train Acc: 93.24% | Test Acc: 87.34%
Model saved to opt_adam_lr0.0001_wd0.0005_bs64_ep50.pth
Test Accura

In [3]:
import glob
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import os
import re

sns.set_style("whitegrid")
plt.figure(figsize=(12, 8))

files = glob.glob("opt_*_lr*_wd*_bs*_ep*-acc.txt")
files = [f for f in files if not f.startswith("confusion_matrix_")]

line_styles = ['-', '--', '-.', ':']
colors = sns.color_palette("husl", n_colors=len(files))

all_data = []

def parse_filename(filename):
    filename = os.path.basename(filename)
    params_str = filename.replace('-acc.txt', '')
    
    pattern = r"opt_([a-zA-Z]+)_lr([\d.e+-]+)_wd([\d.e+-]+)_bs(\d+)_ep(\d+)"
    match = re.match(pattern, params_str)
    
    if match:
        try:
            return {
                'optimizer': match.group(1).lower(),
                'lr': float(match.group(2)),
                'wd': float(match.group(3)),
                'bs': int(match.group(4)),
                'ep': int(match.group(5)),
                'params_str': params_str
            }
        except ValueError as e:
            raise ValueError(f"Failed to convert parameters: {str(e)}")
    raise ValueError("Filename pattern not matched")

for i, file in enumerate(files):
    try:
        params = parse_filename(file)
        if params is None:
            raise ValueError(f"Could not parse parameters from filename: {file}")
        
        data = pd.read_csv(file)
        
        data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_')
        
        required_cols = {'epoch', 'train_acc', 'test_acc'}
        if not required_cols.issubset(set(data.columns)):
            raise ValueError(f"Missing required columns in {file}")
        
        for key, value in params.items():
            data[key] = value
        
        all_data.append(data)
        print(f"Successfully processed: {file}")
        
    except Exception as e:
        print(f"Error processing {file}: {str(e)}")

if all_data:
    combined_data = pd.concat(all_data)
    
    def create_label(row):
        return (f"opt={row['optimizer']}\n"
                f"lr={row['lr']:.0e}\n"
                f"wd={row['wd']:.0e}\n"
                f"bs={row['bs']}\n"
                f"ep={row['ep']}")
    
    combined_data['label'] = combined_data.apply(create_label, axis=1)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 14), sharex=True)
    
    optimizers = combined_data['optimizer'].unique()
    opt_colors = sns.color_palette("Set1", n_colors=len(optimizers))
    color_map = {opt: color for opt, color in zip(optimizers, opt_colors)}
    
    for i, (label, group) in enumerate(combined_data.groupby('label')):
        opt = group['optimizer'].iloc[0]
        ax1.plot(group['epoch'], group['train_acc'],
                label=label,
                linestyle=line_styles[i % len(line_styles)],
                color=color_map[opt],
                linewidth=2,
                alpha=0.8)
        
        ax2.plot(group['epoch'], group['test_acc'],
                label=label,
                linestyle=line_styles[i % len(line_styles)],
                color=color_map[opt],
                linewidth=2,
                alpha=0.8)
    
    ax1.set_title('Training Accuracy Comparison', fontsize=14, pad=10)
    ax1.set_ylabel('Accuracy (%)', fontsize=12)
    ax1.grid(True, alpha=0.3, linestyle='--')
    
    ax2.set_title('Test Accuracy Comparison', fontsize=14, pad=10)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.grid(True, alpha=0.3, linestyle='--')
    
    handles, labels = ax1.get_legend_handles_labels()
    fig.legend(handles, labels, 
              bbox_to_anchor=(1.12, 0.5), 
              loc='center left',
              frameon=True,
              shadow=True,
              fontsize=9,
              title='Hyperparameters',
              title_fontsize=10)
    
    plt.tight_layout()
    
    comparison_filename = "accuracy_comparison_with_optimizer.png"
    plt.savefig(comparison_filename, bbox_inches='tight', dpi=300)
    plt.close()
    
    print(f"\nSuccessfully compared {len(all_data)} configurations")
    print(f"Comparison plot saved to: {comparison_filename}")
    
    combined_data.to_csv("combined_results_with_optimizer.csv", index=False)
    print("Combined results saved to: combined_results_with_optimizer.csv")
    
    for opt in optimizers:
        opt_data = combined_data[combined_data['optimizer'] == opt]
        
        if len(opt_data) > 0:
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12), sharex=True)
            
            for i, (label, group) in enumerate(opt_data.groupby('label')):
                ax1.plot(group['epoch'], group['train_acc'],
                        label=label,
                        linestyle=line_styles[i % len(line_styles)],
                        color=colors[i],
                        linewidth=2,
                        alpha=0.8)
                
                ax2.plot(group['epoch'], group['test_acc'],
                        label=label,
                        linestyle=line_styles[i % len(line_styles)],
                        color=colors[i],
                        linewidth=2,
                        alpha=0.8)
            
            ax1.set_title(f'Training Accuracy Comparison ({opt.upper()})', fontsize=14, pad=10)
            ax1.set_ylabel('Accuracy (%)', fontsize=12)
            ax1.grid(True, alpha=0.3, linestyle='--')
            
            ax2.set_title(f'Test Accuracy Comparison ({opt.upper()})', fontsize=14, pad=10)
            ax2.set_xlabel('Epoch', fontsize=12)
            ax2.set_ylabel('Accuracy (%)', fontsize=12)
            ax2.grid(True, alpha=0.3, linestyle='--')
            
            handles, labels = ax1.get_legend_handles_labels()
            fig.legend(handles, labels, 
                      bbox_to_anchor=(1.12, 0.5), 
                      loc='center left',
                      frameon=True,
                      shadow=True,
                      fontsize=9,
                      title='Hyperparameters',
                      title_fontsize=10)
            
            plt.tight_layout()
            
            opt_filename = f"accuracy_comparison_{opt}.png"
            plt.savefig(opt_filename, bbox_inches='tight', dpi=300)
            plt.close()
            print(f"Optimizer-specific plot saved to: {opt_filename}")
    
else:
    print("\nNo valid data files found for comparison")
    
    if files:
        print("\nFound files (but couldn't process):")
        for f in files:
            print(f" - {f}")
            
        print("\nSample file content:")
        try:
            with open(files[0], 'r') as f:
                print(f.read())
        except:
            print("Could not read file content")

Successfully processed: opt_sgd_lr1e-05_wd0.0005_bs64_ep50-acc.txt
Successfully processed: opt_adam_lr1e-05_wd5e-05_bs128_ep50-acc.txt
Successfully processed: opt_sgd_lr0.0001_wd0.0005_bs64_ep50-acc.txt
Successfully processed: opt_adam_lr0.0001_wd0.0005_bs64_ep50-acc.txt
Successfully processed: opt_sgd_lr0.0001_wd5e-05_bs128_ep50-acc.txt
Successfully processed: opt_sgd_lr1e-05_wd0.0005_bs128_ep50-acc.txt
Successfully processed: opt_adam_lr1e-05_wd0.0005_bs64_ep50-acc.txt
Successfully processed: opt_sgd_lr0.0001_wd0.0005_bs128_ep50-acc.txt
Successfully processed: opt_adam_lr1e-05_wd5e-05_bs64_ep50-acc.txt
Successfully processed: opt_adam_lr0.0001_wd0.0005_bs128_ep50-acc.txt
Successfully processed: opt_sgd_lr1e-05_wd5e-05_bs128_ep50-acc.txt
Successfully processed: opt_adam_lr1e-05_wd0.0005_bs128_ep50-acc.txt
Successfully processed: opt_sgd_lr0.0001_wd5e-05_bs64_ep50-acc.txt
Successfully processed: opt_sgd_lr1e-05_wd5e-05_bs64_ep50-acc.txt
Successfully processed: opt_adam_lr0.0001_wd5e-05

<Figure size 1200x800 with 0 Axes>

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import argparse

class VGG11(nn.Module):
    def __init__(self):
        super(VGG11, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(512*1*1, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

class CIFAR10Classifier:
    def __init__(self, args):
        self.args = args
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.classes = ('plane', 'car', 'bird', 'cat', 'deer', 
                       'dog', 'frog', 'horse', 'ship', 'truck')
        
        self.model = VGG11().to(self.device)
        self.criterion = nn.CrossEntropyLoss()
        
        if args.optimizer.lower() == 'adam':
            self.optimizer = optim.Adam(self.model.parameters(), 
                                      lr=args.lr, 
                                      weight_decay=args.weight_decay)
        elif args.optimizer.lower() == 'sgd':
            self.optimizer = optim.SGD(self.model.parameters(), 
                                      lr=args.lr, 
                                      momentum=0.9,
                                      weight_decay=args.weight_decay)
        else:
            raise ValueError(f"Unsupported optimizer: {args.optimizer}")
        
        self._load_data()

    def _load_data(self):
        transform_train = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ])

        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ])

        trainset = torchvision.datasets.CIFAR10(
            root='./data', train=True, download=True, transform=transform_train)
        self.trainloader = DataLoader(trainset, batch_size=self.args.batch_size,
                                     shuffle=True, num_workers=2)

        testset = torchvision.datasets.CIFAR10(
            root='./data', train=False, download=True, transform=transform_test)
        self.testloader = DataLoader(testset, batch_size=self.args.batch_size,
                                    shuffle=False, num_workers=2)

    def train(self):
        print(f"\nStarting training with {self.args.epochs} epochs...")
        filename = f"optimized_opt_{self.args.optimizer}_lr{self.args.lr}_wd{self.args.weight_decay}_bs{self.args.batch_size}_ep{self.args.epochs}-acc.txt"
        
        with open(filename, 'w') as f:  
            f.write("Epoch, Train_acc, Test_acc\n")     
            
            for epoch in range(self.args.epochs):
                self.model.train()      
                total = 0
                correct = 0
                
                for inputs, targets in self.trainloader:
                    inputs, targets = inputs.to(self.device), targets.to(self.device)
                    
                    self.optimizer.zero_grad()
                    outputs = self.model(inputs)
                    loss = self.criterion(outputs, targets)
                    loss.backward()
                    self.optimizer.step()
                    
                    _, predicted = outputs.max(1)
                    total += targets.size(0)
                    correct += predicted.eq(targets).sum().item()
                
                train_acc = 100 * correct / total
                
                test_acc = self.test()
                
                f.write(f"{epoch}, {train_acc:.2f}, {test_acc:.2f}\n") 
                
                if (epoch+1) % 10 == 0:
                    print(f"Epoch [{epoch+1}/{self.args.epochs}] | "
                          f"Train Acc: {train_acc:.2f}% | "
                          f"Test Acc: {test_acc:.2f}%")
        
        model_save_name = f"optimized_opt_{self.args.optimizer}_lr{self.args.lr}_wd{self.args.weight_decay}_bs{self.args.batch_size}_ep{self.args.epochs}.pth"
        torch.save(self.model.state_dict(), model_save_name)
        print(f"Model saved to {model_save_name}")

    def test(self):
        self.model.eval()
        all_targets = []
        all_preds = []
        with torch.no_grad():
            for inputs, targets in self.testloader:
                inputs, targets = inputs.to(self.device), targets.to(self.device)
                outputs = self.model(inputs)
                _, predicted = outputs.max(1)
                
                all_targets.extend(targets.cpu().numpy())
                all_preds.extend(predicted.cpu().numpy())
        
        accuracy = 100 * np.sum(np.array(all_preds) == np.array(all_targets)) / len(all_targets)
        return accuracy

    def plot_confusion_matrix(self):
        self.model.eval()
        all_targets = []
        all_preds = []
        
        with torch.no_grad():
            for inputs, targets in self.testloader:
                inputs, targets = inputs.to(self.device), targets.to(self.device)
                outputs = self.model(inputs)
                _, predicted = outputs.max(1)
                
                all_targets.extend(targets.cpu().numpy())
                all_preds.extend(predicted.cpu().numpy())
        
        cm = confusion_matrix(all_targets, all_preds)
        plt.figure(figsize=(12, 10))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=self.classes, 
                   yticklabels=self.classes)
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        
        filename = f"optimized_confusion_matrix_opt_{self.args.optimizer}_lr{self.args.lr}_wd{self.args.weight_decay}_bs{self.args.batch_size}_ep{self.args.epochs}.png"
        plt.savefig(filename)
        plt.close() 
        print(f"Confusion matrix saved to {filename}")

In [5]:
from types import SimpleNamespace

param_grid = {
    'batch_size': [64, 128],
    'lr': [ 0.0001, 0.00001],
    'weight_decay': [5e-4, 5e-5],
    'optimizer': ['sgd', 'adam']
}

base_args = {
    'train': True,
    'test': True,
    'plot': True,
    'epochs': 50,
    'save_path': None,
    'load_path': None
}

results = []

for batch_size in param_grid['batch_size']:
    for lr in param_grid['lr']:
        for weight_decay in param_grid['weight_decay']:
            for optimizer in param_grid['optimizer']:
                print(f"\n=== Testing config: batch_size={batch_size}, lr={lr}, "
                      f"weight_decay={weight_decay}, optimizer={optimizer} ===")
                
                current_args = base_args.copy()
                current_args.update({
                    'batch_size': batch_size,
                    'lr': lr,
                    'weight_decay': weight_decay,
                    'optimizer': optimizer,
                    'save_path': f"model_bs{batch_size}_lr{lr}_wd{weight_decay}_{optimizer}.pth"
                })
                
                args = SimpleNamespace(**current_args)
                
                classifier = CIFAR10Classifier(args)
                
                if args.train:
                    classifier.train()
                
                if args.test:
                    accuracy = classifier.test()
                    results.append({
                        'batch_size': batch_size,
                        'lr': lr,
                        'weight_decay': weight_decay,
                        'optimizer': optimizer,
                        'accuracy': accuracy
                    })
                    print(f"Test Accuracy: {accuracy:.2f}%")

if results:
    best_result = max(results, key=lambda x: x['accuracy'])
    print("\n=== Best Configuration ===")
    print(f"Batch Size: {best_result['batch_size']}")
    print(f"Learning Rate: {best_result['lr']}")
    print(f"Weight Decay: {best_result['weight_decay']}")
    print(f"Optimizer: {best_result['optimizer']}")
    print(f"Accuracy: {best_result['accuracy']:.2f}%")

    with open('best_config.txt', 'w') as f:
        f.write(str(best_result))


=== Testing config: batch_size=64, lr=0.0001, weight_decay=0.0005, optimizer=sgd ===
Files already downloaded and verified
Files already downloaded and verified

Starting training with 50 epochs...
Epoch [10/50] | Train Acc: 62.89% | Test Acc: 66.25%
Epoch [20/50] | Train Acc: 74.51% | Test Acc: 75.24%
Epoch [30/50] | Train Acc: 79.76% | Test Acc: 79.41%
Epoch [40/50] | Train Acc: 83.23% | Test Acc: 81.09%
Epoch [50/50] | Train Acc: 86.05% | Test Acc: 82.40%
Model saved to optimized_opt_sgd_lr0.0001_wd0.0005_bs64_ep50.pth
Test Accuracy: 82.40%

=== Testing config: batch_size=64, lr=0.0001, weight_decay=0.0005, optimizer=adam ===
Files already downloaded and verified
Files already downloaded and verified

Starting training with 50 epochs...
Epoch [10/50] | Train Acc: 83.29% | Test Acc: 80.11%
Epoch [20/50] | Train Acc: 88.86% | Test Acc: 84.25%
Epoch [30/50] | Train Acc: 91.69% | Test Acc: 86.07%
Epoch [40/50] | Train Acc: 93.49% | Test Acc: 86.82%
Epoch [50/50] | Train Acc: 94.91% | T

In [6]:
import glob
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import os
import re

sns.set_style("whitegrid")
plt.figure(figsize=(12, 8))

files = glob.glob("optimized_opt_*_lr*_wd*_bs*_ep*-acc.txt")
files = [f for f in files if not f.startswith("optimized_confusion_matrix_")]

line_styles = ['-', '--', '-.', ':']
colors = sns.color_palette("husl", n_colors=len(files))

all_data = []

def parse_filename(filename):
    filename = os.path.basename(filename)
    params_str = filename.replace('-acc.txt', '')
    
    pattern = r"optimized_opt_([a-zA-Z]+)_lr([\d.e+-]+)_wd([\d.e+-]+)_bs(\d+)_ep(\d+)"
    match = re.match(pattern, params_str)
    
    if match:
        try:
            return {
                'optimizer': match.group(1).lower(),
                'lr': float(match.group(2)),
                'wd': float(match.group(3)),
                'bs': int(match.group(4)),
                'ep': int(match.group(5)),
                'params_str': params_str
            }
        except ValueError as e:
            raise ValueError(f"Failed to convert parameters: {str(e)}")
    raise ValueError("Filename pattern not matched")

for i, file in enumerate(files):
    try:
        params = parse_filename(file)
        if params is None:
            raise ValueError(f"Could not parse parameters from filename: {file}")
        
        data = pd.read_csv(file)
        
        data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_')
        
        required_cols = {'epoch', 'train_acc', 'test_acc'}
        if not required_cols.issubset(set(data.columns)):
            raise ValueError(f"Missing required columns in {file}")
        
        for key, value in params.items():
            data[key] = value
        
        all_data.append(data)
        print(f"Successfully processed: {file}")
        
    except Exception as e:
        print(f"Error processing {file}: {str(e)}")

if all_data:
    combined_data = pd.concat(all_data)
    
    def create_label(row):
        return (f"opt={row['optimizer']}\n"
                f"lr={row['lr']:.0e}\n"
                f"wd={row['wd']:.0e}\n"
                f"bs={row['bs']}\n"
                f"ep={row['ep']}")
    
    combined_data['label'] = combined_data.apply(create_label, axis=1)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 14), sharex=True)
    
    optimizers = combined_data['optimizer'].unique()
    opt_colors = sns.color_palette("Set1", n_colors=len(optimizers))
    color_map = {opt: color for opt, color in zip(optimizers, opt_colors)}
    
    for i, (label, group) in enumerate(combined_data.groupby('label')):
        opt = group['optimizer'].iloc[0]
        ax1.plot(group['epoch'], group['train_acc'],
                label=label,
                linestyle=line_styles[i % len(line_styles)],
                color=color_map[opt],
                linewidth=2,
                alpha=0.8)
        
        ax2.plot(group['epoch'], group['test_acc'],
                label=label,
                linestyle=line_styles[i % len(line_styles)],
                color=color_map[opt],
                linewidth=2,
                alpha=0.8)
    
    ax1.set_title('Training Accuracy Comparison', fontsize=14, pad=10)
    ax1.set_ylabel('Accuracy (%)', fontsize=12)
    ax1.grid(True, alpha=0.3, linestyle='--')
    
    ax2.set_title('Test Accuracy Comparison', fontsize=14, pad=10)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.grid(True, alpha=0.3, linestyle='--')
    
    handles, labels = ax1.get_legend_handles_labels()
    fig.legend(handles, labels, 
              bbox_to_anchor=(1.12, 0.5), 
              loc='center left',
              frameon=True,
              shadow=True,
              fontsize=9,
              title='Hyperparameters',
              title_fontsize=10)
    
    plt.tight_layout()
    
    comparison_filename = "optimized_accuracy_comparison_with_optimizer.png"
    plt.savefig(comparison_filename, bbox_inches='tight', dpi=300)
    plt.close()
    
    print(f"\nSuccessfully compared {len(all_data)} configurations")
    print(f"Comparison plot saved to: {comparison_filename}")
    
    combined_data.to_csv("optimized_ombined_results_with_optimizer.csv", index=False)
    print("Combined results saved to: optimized_combined_results_with_optimizer.csv")
    
    for opt in optimizers:
        opt_data = combined_data[combined_data['optimizer'] == opt]
        
        if len(opt_data) > 0:
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12), sharex=True)
            
            for i, (label, group) in enumerate(opt_data.groupby('label')):
                ax1.plot(group['epoch'], group['train_acc'],
                        label=label,
                        linestyle=line_styles[i % len(line_styles)],
                        color=colors[i],
                        linewidth=2,
                        alpha=0.8)
                
                ax2.plot(group['epoch'], group['test_acc'],
                        label=label,
                        linestyle=line_styles[i % len(line_styles)],
                        color=colors[i],
                        linewidth=2,
                        alpha=0.8)
            
            ax1.set_title(f'Training Accuracy Comparison ({opt.upper()})', fontsize=14, pad=10)
            ax1.set_ylabel('Accuracy (%)', fontsize=12)
            ax1.grid(True, alpha=0.3, linestyle='--')
            
            ax2.set_title(f'Test Accuracy Comparison ({opt.upper()})', fontsize=14, pad=10)
            ax2.set_xlabel('Epoch', fontsize=12)
            ax2.set_ylabel('Accuracy (%)', fontsize=12)
            ax2.grid(True, alpha=0.3, linestyle='--')
            
            handles, labels = ax1.get_legend_handles_labels()
            fig.legend(handles, labels, 
                      bbox_to_anchor=(1.12, 0.5), 
                      loc='center left',
                      frameon=True,
                      shadow=True,
                      fontsize=9,
                      title='Hyperparameters',
                      title_fontsize=10)
            
            plt.tight_layout()
            
            opt_filename = f"accuracy_accuracy_comparison_{opt}.png"
            plt.savefig(opt_filename, bbox_inches='tight', dpi=300)
            plt.close()
            print(f"Optimizer-specific plot saved to: {opt_filename}")
    
else:
    print("\nNo valid data files found for comparison")
    
    if files:
        print("\nFound files (but couldn't process):")
        for f in files:
            print(f" - {f}")
            
        print("\nSample file content:")
        try:
            with open(files[0], 'r') as f:
                print(f.read())
        except:
            print("Could not read file content")

Successfully processed: optimized_opt_sgd_lr1e-05_wd0.0005_bs128_ep50-acc.txt
Successfully processed: optimized_opt_adam_lr0.0001_wd0.0005_bs128_ep50-acc.txt
Successfully processed: optimized_opt_sgd_lr1e-05_wd5e-05_bs64_ep50-acc.txt
Successfully processed: optimized_opt_adam_lr1e-05_wd5e-05_bs64_ep50-acc.txt
Successfully processed: optimized_opt_adam_lr0.0001_wd0.0005_bs64_ep50-acc.txt
Successfully processed: optimized_opt_sgd_lr0.0001_wd0.0005_bs64_ep50-acc.txt
Successfully processed: optimized_opt_adam_lr1e-05_wd0.0005_bs64_ep50-acc.txt
Successfully processed: optimized_opt_sgd_lr0.0001_wd5e-05_bs64_ep50-acc.txt
Successfully processed: optimized_opt_adam_lr0.0001_wd5e-05_bs64_ep50-acc.txt
Successfully processed: optimized_opt_adam_lr1e-05_wd5e-05_bs128_ep50-acc.txt
Successfully processed: optimized_opt_adam_lr0.0001_wd5e-05_bs128_ep50-acc.txt
Successfully processed: optimized_opt_sgd_lr1e-05_wd0.0005_bs64_ep50-acc.txt
Successfully processed: optimized_opt_adam_lr1e-05_wd0.0005_bs128

<Figure size 1200x800 with 0 Axes>